In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:

import sys
if 'google.colab' in sys.modules:
    %pip install -q tensorflow scikit-learn
# All imports
import os
import sys
import math
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from typing import List, Sequence, Tuple, Optional, Union

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler

# TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras import utils as np_utils
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Dense, Activation, Dropout, Conv2D, AveragePooling2D, MaxPooling2D,
    SeparableConv2D, DepthwiseConv2D, BatchNormalization, SpatialDropout2D,
    Input, Flatten, GlobalMaxPooling2D
)
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.callbacks import EarlyStopping


In [ ]:

tf.config.list_physical_devices('GPU')


In [ ]:

# Paths & basic config (edit BASE_DIR if running on Colab)
BASE_DIR = '/content/drive/MyDrive/'
LABELED_EEG_DATA_DIR = Path(BASE_DIR) / 'labeled_eeg_data'
PRE_WINDOW_MS = 500
POST_WINDOW_MS = 1000

# --------------------------
# CHANNEL CONFIGURATION
# --------------------------
DEFAULT_EEGNET_CHANNELS = ['F3', 'F4', 'F7', 'F8', 'Fz', 'FC1', 'FC2', 'FC5', 'FC6',
                          'Fp1', 'Fp2', 'Pz', 'P3', 'P4', 'P7', 'P8']

# Auto-discover subjects
USE_SUBJECTS = sorted([
    d.name for d in Path(LABELED_EEG_DATA_DIR).iterdir()
    if d.is_dir() and d.name.startswith('P')
])
print(f"Found {len(USE_SUBJECTS)} subjects: {USE_SUBJECTS[:10]} ...")

CHANNELS_TO_USE = DEFAULT_EEGNET_CHANNELS
print(f"Using {len(CHANNELS_TO_USE)} channels: {CHANNELS_TO_USE}")



In [ ]:
from collections import Counter

# --------------------------
# EEG DATA LOADING
# --------------------------
def load_eeg_data(subjects, channels, pre_ms, post_ms, epoch_start_ms=-200, epoch_end_ms=600, fs=500):
    """
    Load EEG data for specified subjects and channels.
    
    Data is loaded from [-500, 1000] ms window, then a specific epoch is extracted.
    
    Args:
        subjects: list of subject IDs
        channels: list of channel names
        pre_ms: pre-fixation window in file (e.g., 500)
        post_ms: post-fixation window in file (e.g., 1000)
        epoch_start_ms: epoch start relative to fixation (default: -200)
        epoch_end_ms: epoch end relative to fixation (default: 600)
        fs: sampling frequency in Hz (default: 500)
    """
    all_X, all_y, all_subjects = [], [], []
    
    # Calculate total time points in loaded data
    total_time_points = int((pre_ms + post_ms) / 1000 * fs)
    fixation_idx = int(pre_ms / 1000 * fs)  # Index of fixation point
    
    # Calculate epoch indices
    epoch_start_idx = fixation_idx + int(epoch_start_ms / 1000 * fs)
    epoch_end_idx = fixation_idx + int(epoch_end_ms / 1000 * fs)
    
    print(f"Loading data from [-{pre_ms}, {post_ms}] ms window")
    print(f"Extracting epoch [{epoch_start_ms}, {epoch_end_ms}] ms (indices {epoch_start_idx}:{epoch_end_idx})")

    for subject in subjects:
        file_path = LABELED_EEG_DATA_DIR / subject / f"{pre_ms}_{post_ms}_training_data.npz"
        if not file_path.exists():
            continue

        # Load with memory mapping first to check channels
        with np.load(file_path, mmap_mode='r') as data:
            ch_names = data['channel_names']

            # Find indices of channels we want
            ch_indices = []
            for ch in channels:
                if ch in ch_names:
                    ch_indices.append(list(ch_names).index(ch))

            if len(ch_indices) == 0:
                continue  # Skip if no channels found

            # Only load the data we need - much more memory efficient
            X_full = data['X']  # Still memory mapped
            y = data['y'][:]    # Load labels (small)

            # Extract specific epoch and channels
            X_selected = X_full[:, ch_indices, epoch_start_idx:epoch_end_idx].copy().astype(np.float32)

        all_X.append(X_selected)
        all_y.append(y)
        all_subjects.extend([subject] * len(y))  # Track which subject each sample belongs to

        # Force garbage collection to free memory
        del X_selected

    # Combine all subjects
    X_combined = np.concatenate(all_X, axis=0).astype(np.float32)
    y_combined = np.concatenate(all_y, axis=0)

    print(f"\nLoaded {X_combined.shape[0]} trials from {len(set(all_subjects))} subjects")
    print(f"Using {len(ch_indices)} channels, {X_combined.shape[2]} time points")
    print(f"Epoch duration: {(epoch_end_ms - epoch_start_ms)} ms")

    # Add label distribution analysis
    label_counts = Counter(y_combined)
    total_samples = len(y_combined)
    print("\nLabel Distribution:")
    for label, count in sorted(label_counts.items()):
        proportion = count / total_samples
        print(f"  Label {label}: {count} samples ({proportion:.2%})")

    return X_combined, y_combined, np.array(all_subjects)



In [ ]:

# --------------------------
# PER-SUBJECT NORMALIZATION
# --------------------------
def normalize_per_subject(X, y, subjects):
    """
    Normalize each subject's data individually using StandardScaler.
    
    Args:
        X: (samples, channels, time_points)
        y: (samples,)
        subjects: (samples,) array of subject IDs
    
    Returns:
        X_normalized: normalized data
        scalers: dict of StandardScaler objects per subject
    """
    X_normalized = X.copy()
    scalers = {}
    
    unique_subjects = np.unique(subjects)
    for subject in unique_subjects:
        mask = subjects == subject
        X_subject = X[mask]
        
        # Fit scaler on this subject's data
        scaler = StandardScaler()
        X_reshaped = X_subject.reshape(X_subject.shape[0], -1)
        scaler.fit(X_reshaped)
        
        # Transform this subject's data
        X_normalized[mask] = scaler.transform(X_reshaped).reshape(X_subject.shape)
        scalers[subject] = scaler
        
        print(f"  {subject}: mean={scaler.mean_.mean():.4f}, std={scaler.scale_.mean():.4f}")
    
    return X_normalized, scalers




In [ ]:

# --------------------------
# METHOD 1: NORMAL RANDOM SPLIT WITH PER-SUBJECT NORMALIZATION
# --------------------------
def load_data_normal_split(test_size=0.2, val_size=0.2, seed=42):
    """
    Normal random train/val/test split with per-subject normalization.
    This method mixes subjects but normalizes each subject individually before splitting.
    """
    print("\n" + "="*70)
    print("LOADING DATA: NORMAL RANDOM SPLIT (per-subject normalization)")
    print("="*70)
    
    # Load all data with epoch extraction
    X_all, y_all, subjects_all = load_eeg_data(USE_SUBJECTS, CHANNELS_TO_USE, PRE_WINDOW_MS, POST_WINDOW_MS,
                                               epoch_start_ms=-200, epoch_end_ms=600, fs=500)
    
    # Normalize each subject individually
    print("\nNormalizing per subject:")
    X_normalized, scalers = normalize_per_subject(X_all, y_all, subjects_all)
    
    # Add EEGNet dimension
    X_normalized = X_normalized[..., np.newaxis]  # (N, Chans, Time, 1)
    
    # Random train/val/test split (mixes subjects)
    print("\nPerforming random train/val/test split...")
    X_train, X_test, y_train, y_test, subj_train, subj_test = train_test_split(
        X_normalized, y_all, subjects_all, test_size=test_size, random_state=seed, stratify=y_all
    )
    
    X_train, X_val, y_train, y_val, subj_train, subj_val = train_test_split(
        X_train, y_train, subj_train, test_size=val_size/(1-test_size), random_state=seed, stratify=y_train
    )
    
    # One-hot encode labels
    y_train_cat = tf.keras.utils.to_categorical(y_train.astype(int), 2)
    y_val_cat = tf.keras.utils.to_categorical(y_val.astype(int), 2)
    y_test_cat = tf.keras.utils.to_categorical(y_test.astype(int), 2)
    
    print(f"\nData shapes:")
    print(f"  Train: {X_train.shape} from {len(np.unique(subj_train))} subjects")
    print(f"  Val:   {X_val.shape} from {len(np.unique(subj_val))} subjects")
    print(f"  Test:  {X_test.shape} from {len(np.unique(subj_test))} subjects")
    
    return X_train, X_val, X_test, y_train_cat, y_val_cat, y_test_cat




In [ ]:

# --------------------------
# METHOD 2: LEAVE-ONE-OUT CROSS-VALIDATION (N subjects left out)
# --------------------------
def load_data_loocv(n_test_subjects=3, seed=42):
    """
    Leave-one-out style CV: N subjects are held out for testing, 
    remaining subjects are split into train/val (80/20).
    Normalization is done per-subject before any splitting.
    
    Args:
        n_test_subjects: number of subjects to hold out for testing
        seed: random seed
    
    Returns:
        List of tuples (X_train, X_val, X_test, y_train, y_val, y_test, test_subjects)
        One fold per iteration, cycling through different subject combinations.
    """
    print("\n" + "="*70)
    print(f"LOADING DATA: LEAVE-{n_test_subjects}-OUT CV (per-subject normalization)")
    print("="*70)
    
    # Load all data with epoch extraction
    X_all, y_all, subjects_all = load_eeg_data(USE_SUBJECTS, CHANNELS_TO_USE, PRE_WINDOW_MS, POST_WINDOW_MS,
                                               epoch_start_ms=-200, epoch_end_ms=600, fs=500)
    
    # Normalize each subject individually
    print("\nNormalizing per subject:")
    X_normalized, scalers = normalize_per_subject(X_all, y_all, subjects_all)
    
    # Add EEGNet dimension
    X_normalized = X_normalized[..., np.newaxis]  # (N, Chans, Time, 1)
    
    # Create folds
    unique_subjects = np.unique(subjects_all)
    n_subjects = len(unique_subjects)
    
    print(f"\nCreating folds with {n_test_subjects} test subjects each...")
    print(f"Total subjects: {n_subjects}")
    
    folds = []
    np.random.seed(seed)
    
    # Shuffle subjects and create multiple folds
    shuffled_subjects = np.random.permutation(unique_subjects)
    
    for fold_idx in range(n_subjects // n_test_subjects):
        # Select test subjects
        test_subject_indices = np.arange(fold_idx * n_test_subjects, 
                                         (fold_idx + 1) * n_test_subjects)
        test_subjects = shuffled_subjects[test_subject_indices]
        train_val_subjects = np.setdiff1d(shuffled_subjects, test_subjects)
        
        # Split data by subject
        test_mask = np.isin(subjects_all, test_subjects)
        train_val_mask = np.isin(subjects_all, train_val_subjects)
        
        X_test = X_normalized[test_mask]
        y_test = y_all[test_mask]
        
        X_train_val = X_normalized[train_val_mask]
        y_train_val = y_all[train_val_mask]
        subj_train_val = subjects_all[train_val_mask]
        
        # Further split train_val into train/val (80/20)
        X_train, X_val, y_train, y_val, subj_train, subj_val = train_test_split(
            X_train_val, y_train_val, subj_train_val, 
            test_size=0.2, random_state=seed, stratify=y_train_val
        )
        
        # One-hot encode
        y_train_cat = tf.keras.utils.to_categorical(y_train.astype(int), 2)
        y_val_cat = tf.keras.utils.to_categorical(y_val.astype(int), 2)
        y_test_cat = tf.keras.utils.to_categorical(y_test.astype(int), 2)
        
        print(f"\nFold {fold_idx + 1}:")
        print(f"  Test subjects: {test_subjects}")
        print(f"  Train: {X_train.shape} from {len(np.unique(subj_train))} subjects")
        print(f"  Val:   {X_val.shape} from {len(np.unique(subj_val))} subjects")
        print(f"  Test:  {X_test.shape}")
        
        folds.append({
            'X_train': X_train,
            'X_val': X_val,
            'X_test': X_test,
            'y_train': y_train_cat,
            'y_val': y_val_cat,
            'y_test': y_test_cat,
            'test_subjects': test_subjects,
            'fold_idx': fold_idx
        })
    
    return folds




In [ ]:


# --------------------------
# EEGNet MODEL
# --------------------------
def EEGNet(nb_classes, Chans, Samples, dropoutRate=0.2, kernLength=64, F1=8, D=3, F2=16, norm_rate=0.25):
    """EEGNet model as per original implementation."""
    input1 = Input(shape=(Chans, Samples, 1))
    block1 = Conv2D(F1, (1, kernLength), padding='same', use_bias=False)(input1)
    block1 = BatchNormalization()(block1)
    block1 = DepthwiseConv2D((Chans, 1), use_bias=False, depth_multiplier=D, depthwise_constraint=max_norm(1.))(block1)
    block1 = BatchNormalization()(block1)
    block1 = Activation('elu')(block1)
    block1 = AveragePooling2D((1, 4))(block1)
    block1 = Dropout(dropoutRate)(block1)

    block2 = SeparableConv2D(F2, (1, 16), use_bias=False, padding='same')(block1)
    block2 = BatchNormalization()(block2)
    block2 = Activation('elu')(block2)
    block2 = AveragePooling2D((1, 8))(block2)
    block2 = Dropout(dropoutRate)(block2)

    flatten = Flatten(name='flatten')(block2)
    dense = Dense(nb_classes, name='dense', kernel_constraint=max_norm(norm_rate))(flatten)
    softmax = Activation('softmax', name='softmax')(dense)
    return Model(inputs=input1, outputs=softmax)



# --------------------------
# IMPROVED CNN MODEL
# --------------------------
def build_simple_cnn(Chans, Samples):
    """Simple CNN with batch norm and global pooling."""
    model = Sequential([
        Conv2D(16, (1, 10), padding='same', use_bias=False, input_shape=(Chans, Samples, 1)),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling2D((1, 4)),

        Conv2D(32, (1, 5), padding='same', use_bias=False),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling2D((1, 4)),
        Dropout(0.5),

        GlobalMaxPooling2D(),
        Dense(32, activation='relu'),
        Dropout(0.5),
        Dense(2, activation='softmax')
    ])
    return model


In [ ]:



# --------------------------
# TRAINING AND EVALUATION
# --------------------------
class TrainingHistory:
    """Track training history for plotting."""
    def __init__(self):
        self.history = {}
    
    def set_history(self, h):
        self.history = h

def train_and_evaluate_model(model, X_train, X_val, X_test, y_train, y_val, y_test, 
                            model_name, epochs=30, batch_size=32, patience=8):
    """
    Train model and return metrics + history.
    
    Returns:
        dict with keys: train_bal_acc, val_bal_acc, test_bal_acc, train_f1, val_f1, test_f1, history
    """
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
    early_stop = EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)
    
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=1
    )
    
    # Get predictions for balanced accuracy and F1
    y_train_pred = np.argmax(model.predict(X_train, verbose=0), axis=1)
    y_val_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)
    y_test_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    
    y_train_true = np.argmax(y_train, axis=1)
    y_val_true = np.argmax(y_val, axis=1)
    y_test_true = np.argmax(y_test, axis=1)
    
    # Compute metrics
    train_bal_acc = balanced_accuracy_score(y_train_true, y_train_pred)
    val_bal_acc = balanced_accuracy_score(y_val_true, y_val_pred)
    test_bal_acc = balanced_accuracy_score(y_test_true, y_test_pred)
    
    train_f1 = f1_score(y_train_true, y_train_pred, average='macro')
    val_f1 = f1_score(y_val_true, y_val_pred, average='macro')
    test_f1 = f1_score(y_test_true, y_test_pred, average='macro')
    
    print(f"\n  {model_name}:")
    print(f"    Balanced Accuracy - Train: {train_bal_acc:.3f} | Val: {val_bal_acc:.3f} | Test: {test_bal_acc:.3f}")
    print(f"    F1 Macro          - Train: {train_f1:.3f} | Val: {val_f1:.3f} | Test: {test_f1:.3f}")
    
    return {
        'train_bal_acc': train_bal_acc,
        'val_bal_acc': val_bal_acc,
        'test_bal_acc': test_bal_acc,
        'train_f1': train_f1,
        'val_f1': val_f1,
        'test_f1': test_f1,
        'history': history
    }



In [ ]:


# --------------------------
# PLOTTING LOSS CURVES
# --------------------------
def plot_loss_curves(history_dict, model_names, eval_method, fold_idx=None, plot_individual=False):
    """
    Plot loss and accuracy curves for multiple models.
    
    Args:
        history_dict: dict of {model_name: keras.History}
        model_names: list of model names
        eval_method: 'normal' or 'loocv'
        fold_idx: fold index (for LOOCV), None for normal split
        plot_individual: if True, plot each model separately
    """
    if plot_individual:
        # Plot each model individually
        for model_name, history in history_dict.items():
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            
            axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2, color='#1f77b4')
            axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2, linestyle='--', color='#ff7f0e')
            axes[0].set_xlabel('Epoch', fontsize=12)
            axes[0].set_ylabel('Loss', fontsize=12)
            axes[0].set_title(f'Training Loss - {model_name}', fontsize=13, fontweight='bold')
            axes[0].legend(fontsize=10)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2, color='#1f77b4')
            axes[1].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2, linestyle='--', color='#ff7f0e')
            axes[1].set_xlabel('Epoch', fontsize=12)
            axes[1].set_ylabel('Accuracy', fontsize=12)
            axes[1].set_title(f'Training Accuracy - {model_name}', fontsize=13, fontweight='bold')
            axes[1].legend(fontsize=10)
            axes[1].grid(True, alpha=0.3)
            
            title = f'{model_name} ({eval_method})'
            if fold_idx is not None:
                title += f' - Fold {fold_idx}'
            fig.suptitle(title, fontsize=14, fontweight='bold')
            
            plt.tight_layout()
            plt.show()
    else:
        # Plot all models together
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        for model_name, history in history_dict.items():
            axes[0].plot(history.history['loss'], label=f'{model_name} (train)', alpha=0.7, linewidth=2)
            axes[0].plot(history.history['val_loss'], label=f'{model_name} (val)', linestyle='--', alpha=0.7, linewidth=2)
            
            axes[1].plot(history.history['accuracy'], label=f'{model_name} (train)', alpha=0.7, linewidth=2)
            axes[1].plot(history.history['val_accuracy'], label=f'{model_name} (val)', linestyle='--', alpha=0.7, linewidth=2)
        
        axes[0].set_xlabel('Epoch', fontsize=12)
        axes[0].set_ylabel('Loss', fontsize=12)
        axes[0].set_title(f'Training Loss ({eval_method})', fontsize=13, fontweight='bold')
        axes[0].legend(fontsize=9)
        axes[0].grid(True, alpha=0.3)
        
        axes[1].set_xlabel('Epoch', fontsize=12)
        axes[1].set_ylabel('Accuracy', fontsize=12)
        axes[1].set_title(f'Training Accuracy ({eval_method})', fontsize=13, fontweight='bold')
        axes[1].legend(fontsize=9)
        axes[1].grid(True, alpha=0.3)
        
        if fold_idx is not None:
            fig.suptitle(f'{eval_method.upper()} - Fold {fold_idx}', fontsize=14, fontweight='bold')
        else:
            fig.suptitle(f'{eval_method.upper()}', fontsize=14, fontweight='bold')
        
        plt.tight_layout()
        plt.show()



In [ ]:


# --------------------------
# RESULTS TRACKING
# --------------------------
class ResultsTracker:
    """Track results across multiple runs/folds."""
    def __init__(self):
        self.results = {
            'normal': {'Model': [], 'Train_BalAcc': [], 'Val_BalAcc': [], 'Test_BalAcc': [], 
                      'Train_F1': [], 'Val_F1': [], 'Test_F1': []},
            'loocv': {'Model': [], 'Train_BalAcc': [], 'Val_BalAcc': [], 'Test_BalAcc': [], 
                     'Train_F1': [], 'Val_F1': [], 'Test_F1': [], 'Fold': []}
        }
    
    def record(self, eval_method, model_name, train_bal_acc, val_bal_acc, test_bal_acc, 
               train_f1, val_f1, test_f1, fold_idx=None):
        """Record a single model's results."""
        self.results[eval_method]['Model'].append(model_name)
        self.results[eval_method]['Train_BalAcc'].append(train_bal_acc)
        self.results[eval_method]['Val_BalAcc'].append(val_bal_acc)
        self.results[eval_method]['Test_BalAcc'].append(test_bal_acc)
        self.results[eval_method]['Train_F1'].append(train_f1)
        self.results[eval_method]['Val_F1'].append(val_f1)
        self.results[eval_method]['Test_F1'].append(test_f1)
        
        if fold_idx is not None:
            self.results[eval_method]['Fold'].append(fold_idx)
    
    def get_dataframe(self, eval_method):
        """Get results as a pandas DataFrame."""
        df = pd.DataFrame(self.results[eval_method])
        return df.round(3)
    
    def print_summary(self, eval_method):
        """Print summary statistics."""
        df = self.get_dataframe(eval_method)
        print(f"\n{eval_method.upper()} RESULTS:")
        print(df.to_string(index=False))
        
        if len(df) > 0:
            print(f"\nBalanced Accuracy:")
            print(f"  Avg Test: {df['Test_BalAcc'].mean():.1%} | Best: {df['Test_BalAcc'].max():.1%}")
            print(f"F1 Macro:")
            print(f"  Avg Test: {df['Test_F1'].mean():.1%} | Best: {df['Test_F1'].max():.1%}")




In [ ]:

# =============================================================================
# MAIN EXECUTION
# =============================================================================

tracker = ResultsTracker()

# =============================================================================
# 1. NORMAL RANDOM SPLIT
# =============================================================================
print("\n\n" + "█"*70)
print("█" + " "*68 + "█")
print("█" + " "*15 + "METHOD 1: NORMAL RANDOM SPLIT" + " "*23 + "█")
print("█" + " "*68 + "█")
print("█"*70)

X_train, X_val, X_test, y_train, y_val, y_test = load_data_normal_split()

Chans, Samples = X_train.shape[1], X_train.shape[2]
histories_normal = {}

print("\n" + "="*70)
print("Training models on NORMAL SPLIT...")
print("="*70)


In [ ]:

# Random Forest
print("\n--- Random Forest ---")

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train.reshape(X_train.shape[0], -1), np.argmax(y_train, axis=1))

y_train_pred = rf.predict(X_train.reshape(X_train.shape[0], -1))
y_val_pred = rf.predict(X_val.reshape(X_val.shape[0], -1))
y_test_pred = rf.predict(X_test.reshape(X_test.shape[0], -1))

y_train_true = np.argmax(y_train, axis=1)
y_val_true = np.argmax(y_val, axis=1)
y_test_true = np.argmax(y_test, axis=1)

train_bal_acc = balanced_accuracy_score(y_train_true, y_train_pred)
val_bal_acc = balanced_accuracy_score(y_val_true, y_val_pred)
test_bal_acc = balanced_accuracy_score(y_test_true, y_test_pred)

train_f1 = f1_score(y_train_true, y_train_pred, average='macro')
val_f1 = f1_score(y_val_true, y_val_pred, average='macro')
test_f1 = f1_score(y_test_true, y_test_pred, average='macro')

print(f"  Random Forest:")
print(f"    Balanced Accuracy - Train: {train_bal_acc:.3f} | Val: {val_bal_acc:.3f} | Test: {test_bal_acc:.3f}")
print(f"    F1 Macro          - Train: {train_f1:.3f} | Val: {val_f1:.3f} | Test: {test_f1:.3f}")
tracker.record('normal', 'Random Forest', train_bal_acc, val_bal_acc, test_bal_acc, train_f1, val_f1, test_f1)



In [ ]:

# Simple CNN
print("\n--- Simple CNN ---")
model_cnn = build_simple_cnn(Chans, Samples)
results_cnn = train_and_evaluate_model(model_cnn, X_train, X_val, X_test, y_train, y_val, y_test, 
                                       "Simple CNN", epochs=30, batch_size=32)
histories_normal['Simple CNN'] = results_cnn['history']
tracker.record('normal', 'Simple CNN', results_cnn['train_bal_acc'], results_cnn['val_bal_acc'], results_cnn['test_bal_acc'],
               results_cnn['train_f1'], results_cnn['val_f1'], results_cnn['test_f1'])

# Plot Simple CNN loss curves
plot_loss_curves({'Simple CNN': results_cnn['history']}, ['Simple CNN'], 'normal', plot_individual=True)

del model_cnn


In [ ]:

# EEGNet
print("\n--- EEGNet ---")
model_eegnet = EEGNet(nb_classes=2, Chans=Chans, Samples=Samples, dropoutRate=0.3, kernLength=64, F1=8, D=2, F2=16)
results_eegnet = train_and_evaluate_model(model_eegnet, X_train, X_val, X_test, y_train, y_val, y_test, 
                                          "EEGNet", epochs=30, batch_size=32)
histories_normal['EEGNet'] = results_eegnet['history']
tracker.record('normal', 'EEGNet', results_eegnet['train_bal_acc'], results_eegnet['val_bal_acc'], results_eegnet['test_bal_acc'],
               results_eegnet['train_f1'], results_eegnet['val_f1'], results_eegnet['test_f1'])
del model_eegnet


In [ ]:

# Plot loss curves for normal split
plot_loss_curves(histories_normal, list(histories_normal.keys()), 'normal')

# Print summary
tracker.print_summary('normal')


In [ ]:

# Clean up - Delete all normal split data
del X_train, X_val, X_test, y_train, y_val, y_test
del histories_normal, results_cnn, results_eegnet

# Force garbage collection to free RAM
import gc
gc.collect()



In [ ]:

# =============================================================================
# CELL 1: Load LOOCV data for fold 0
# =============================================================================
print("\n\n" + "█"*70)
print("█" + " "*68 + "█")
print("█" + " "*12 + "METHOD 2: LEAVE-N-OUT CROSS-VALIDATION" + " "*16 + "█")
print("█" + " "*68 + "█")
print("█"*70)

folds = load_data_loocv(n_test_subjects=3, seed=42)

# Extract fold 0
fold_data = folds[0]
fold_idx = fold_data['fold_idx']
test_subjects = fold_data['test_subjects']
X_train = fold_data['X_train']
X_val = fold_data['X_val']
X_test = fold_data['X_test']
y_train = fold_data['y_train']
y_val = fold_data['y_val']
y_test = fold_data['y_test']

# Delete all other folds to free RAM
del folds, fold_data
gc.collect()

Chans, Samples = X_train.shape[1], X_train.shape[2]

print(f"\n{'='*70}")
print(f"FOLD {fold_idx + 1} (Test subjects: {test_subjects})")
print(f"{'='*70}")
print(f"Train shape: {X_train.shape} | Val shape: {X_val.shape} | Test shape: {X_test.shape}")

histories_loocv = {}


In [ ]:


# =============================================================================
# CELL 2: Train Random Forest on Fold 0
# =============================================================================
print("\n--- Random Forest ---")

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train.reshape(X_train.shape[0], -1), np.argmax(y_train, axis=1))

y_train_pred = rf.predict(X_train.reshape(X_train.shape[0], -1))
y_val_pred = rf.predict(X_val.reshape(X_val.shape[0], -1))
y_test_pred = rf.predict(X_test.reshape(X_test.shape[0], -1))

y_train_true = np.argmax(y_train, axis=1)
y_val_true = np.argmax(y_val, axis=1)
y_test_true = np.argmax(y_test, axis=1)

train_bal_acc_rf = balanced_accuracy_score(y_train_true, y_train_pred)
val_bal_acc_rf = balanced_accuracy_score(y_val_true, y_val_pred)
test_bal_acc_rf = balanced_accuracy_score(y_test_true, y_test_pred)

train_f1_rf = f1_score(y_train_true, y_train_pred, average='macro')
val_f1_rf = f1_score(y_val_true, y_val_pred, average='macro')
test_f1_rf = f1_score(y_test_true, y_test_pred, average='macro')

print(f"  Random Forest:")
print(f"    Balanced Accuracy - Train: {train_bal_acc_rf:.3f} | Val: {val_bal_acc_rf:.3f} | Test: {test_bal_acc_rf:.3f}")
print(f"    F1 Macro          - Train: {train_f1_rf:.3f} | Val: {val_f1_rf:.3f} | Test: {test_f1_rf:.3f}")
tracker.record('loocv', 'Random Forest', train_bal_acc_rf, val_bal_acc_rf, test_bal_acc_rf, 
               train_f1_rf, val_f1_rf, test_f1_rf, fold_idx)



In [ ]:


# =============================================================================
# CELL 3: Train Simple CNN on Fold 0
# =============================================================================
print("\n--- Simple CNN ---")

model_cnn = build_simple_cnn(Chans, Samples)
results_cnn = train_and_evaluate_model(
    model_cnn, X_train, X_val, X_test, y_train, y_val, y_test, 
    "Simple CNN", epochs=30, batch_size=32
)
histories_loocv['Simple CNN'] = results_cnn['history']
tracker.record('loocv', 'Simple CNN', results_cnn['train_bal_acc'], results_cnn['val_bal_acc'], results_cnn['test_bal_acc'],
               results_cnn['train_f1'], results_cnn['val_f1'], results_cnn['test_f1'], fold_idx)

# Plot Simple CNN loss curves
plot_loss_curves({'Simple CNN': results_cnn['history']}, ['Simple CNN'], 'loocv', fold_idx=fold_idx, plot_individual=True)

del model_cnn


In [ ]:



# =============================================================================
# CELL 4: Train EEGNet on Fold 0
# =============================================================================
print("\n--- EEGNet ---")

model_eegnet = EEGNet(nb_classes=2, Chans=Chans, Samples=Samples, dropoutRate=0.3, kernLength=64, F1=8, D=2, F2=16)
results_eegnet = train_and_evaluate_model(
    model_eegnet, X_train, X_val, X_test, y_train, y_val, y_test, 
    "EEGNet", epochs=30, batch_size=32
)
histories_loocv['EEGNet'] = results_eegnet['history']
tracker.record('loocv', 'EEGNet', results_eegnet['train_bal_acc'], results_eegnet['val_bal_acc'], results_eegnet['test_bal_acc'],
               results_eegnet['train_f1'], results_eegnet['val_f1'], results_eegnet['test_f1'], fold_idx)

del model_eegnet




In [ ]:

# =============================================================================
# CELL 5: Plot loss curves for Fold 0
# =============================================================================
plot_loss_curves(histories_loocv, list(histories_loocv.keys()), 'loocv', fold_idx=fold_idx)



In [ ]:


# =============================================================================
# CELL 6: Summary for Fold 0
# =============================================================================
print("\n" + "="*70)
print(f"FOLD {fold_idx + 1} SUMMARY")
print("="*70)

df_fold = tracker.get_dataframe('loocv')
df_fold_current = df_fold[df_fold['Fold'] == fold_idx]
print(df_fold_current.to_string(index=False))

print(f"\nTest subjects: {test_subjects}")
avg_test = df_fold_current['Test'].mean()
avg_gap = df_fold_current['Gap'].mean()
print(f"Avg Test Accuracy: {avg_test:.1%}")
print(f"Avg Train-Test Gap: {avg_gap:.1%}")


In [ ]:

# =============================================================================
# FINAL SUMMARY
# =============================================================================
print("\n\n" + "█"*70)
print("█" + " "*68 + "█")
print("█" + " "*22 + "FINAL SUMMARY" + " "*33 + "█")
print("█" + " "*68 + "█")
print("█"*70)

tracker.print_summary('normal')
tracker.print_summary('loocv')

# Compare methods
print("\n" + "="*70)
print("COMPARISON: NORMAL SPLIT vs LEAVE-N-OUT CV")
print("="*70)

df_normal = tracker.get_dataframe('normal')
df_loocv = tracker.get_dataframe('loocv')

if len(df_normal) > 0 and len(df_loocv) > 0:
    print(f"\nNormal Split:")
    print(f"  Avg Test Balanced Accuracy: {df_normal['Test_BalAcc'].mean():.1%}")
    print(f"  Avg Test F1 Macro: {df_normal['Test_F1'].mean():.1%}")
    
    print(f"\nLeave-N-Out CV:")
    print(f"  Avg Test Balanced Accuracy: {df_loocv['Test_BalAcc'].mean():.1%}")
    print(f"  Avg Test F1 Macro: {df_loocv['Test_F1'].mean():.1%}")
    
    diff_bal = df_normal['Test_BalAcc'].mean() - df_loocv['Test_BalAcc'].mean()
    if diff_bal > 0.05:
        print(f"\n⚠️  Normal split shows {diff_bal:.1%} higher balanced accuracy (possible overfitting)")
    else:
        print(f"\n✅ Similar performance across both evaluation methods")